In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
        
# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/iae/pytorch/default/1/iae_steer_vec_dim_128.pt
/kaggle/input/iae/pytorch/default/1/iae_model.pt


In [2]:
    !pip install pyvene transformers torch nnsight FrEIA

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.6/74.6 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.0/105.0 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.7/59.7 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.6/79.6 kB 8.1 MB/s eta 0:00:00
  Created wheel for FrEIA: filename=FrEIA-0.2-py3-none-any.whl size=42763 sha256=d095dca1cebe0530be080a9bbf0a03a99dcbe7e6cacee401a75b3a68bf6d7fd0
  Stored in directory: /root/.cache/pip/wheels/ae/40/63/f30f17a00a5c53f982c5c222995b53fe1ee510d2ea13b00856
Successfully built FrEIA


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pyvene as pv
from FrEIA.framework import SequenceINN
from FrEIA.modules import AllInOneBlock
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import login
login(new_session = False)

2026-02-04 14:44:12.079638: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770216252.265097      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770216252.318124      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770216252.750197      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770216252.750238      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770216252.750241      55 computation_placer.cc:177] computation placer alr

In [4]:
steer_vec_dim = 128
vector_path = f"/kaggle/input/iae/pytorch/default/1/iae_steer_vec_dim_{steer_vec_dim}.pt"
print(f"Loading vector from {vector_path}...")

vector_dict = torch.load(vector_path)
steer_vec = vector_dict["vector"].to(torch.float32).cuda() # Ensure Float32
layer_idx = vector_dict['layer_idx']
model_id = vector_dict['model_id']

model_path = f"/kaggle/input/iae/pytorch/default/1/iae_model.pt"
print(f"Loading model from {model_path}...")
model_dict = torch.load(model_path)
model_state_dict = model_dict['model_state_dict']

Loading vector from /kaggle/input/iae/pytorch/default/1/iae_steer_vec_dim_128.pt...
Loading model from /kaggle/input/iae/pytorch/default/1/iae_model.pt...


In [5]:
def subnet_fc(dims_in, dims_out):
    """Sub-network for coupling layers: standard MLP with non-linearities."""
    return nn.Sequential(
        nn.Linear(dims_in, 128), 
        nn.GELU(),
        nn.Linear(128, dims_out)
    )

class InvertibleUncertaintyAE(nn.Module):
    def __init__(self, input_dim=2304, bottleneck_dim=128):
        super().__init__()
        self.input_dim = input_dim
        self.bottleneck_dim = bottleneck_dim
        
        # 1. Define the Invertible Neural Network (INN)
        # We use FrEIA to build a flow that is mathematically invertible by construction.
        self.inn = SequenceINN(input_dim)
        for _ in range(4): # Stack 4 coupling blocks
            self.inn.append(AllInOneBlock, subnet_constructor=subnet_fc, permute_soft=False)
            
        # 2. Semantic Entropy Probe (Binary Classifier)
        # Changed to output logits for Binary Classification
        self.se_probe = nn.Sequential(
            nn.Linear(bottleneck_dim, 1), # Outputs raw logits (no Sigmoid here, use BCEWithLogitsLoss)
        )

    def forward(self, x):
        # --- Encoder Path ---
        # Map input x to latent space z (bijective)
        z, log_det_jac = self.inn(x)
        
        # Split z into the bottleneck (informative) and residuals (noise)
        y = z[:, :self.bottleneck_dim]
        z_res = z[:, self.bottleneck_dim:]
        
        # --- Decoder / Reconstruction Path ---
        # We force the reconstruction to rely ONLY on y by zeroing out z_res.
        # This acts as the "bottleneck" constraint.
        z_bottlenecked = torch.cat([y, torch.zeros_like(z_res)], dim=1)
        
        # Invert the INN to get the reconstruction
        x_recon, _ = self.inn(z_bottlenecked, rev=True)
        
        # --- Classification Path ---
        # Predict binary uncertainty from the informative bottleneck y
        se_logits = self.se_probe(y)
        
        return x_recon, se_logits, z_res, log_det_jac

    def predict_uncertainty(self, x):
        """Helper for inference to get actual probabilities/labels"""
        self.eval()
        with torch.no_grad():
            _, logits, _, _ = self.forward(x)
            probs = torch.sigmoid(logits)
            return probs > 0.5  # Returns Boolean tensor

In [6]:
class LatentSpaceIntervention(nn.Module):
    def __init__(self, iae_model, steering_vec, strength=1.0):
        super().__init__()
        self.iae = iae_model
        
        # 1. Store IAE metadata for casting later
        # We need to know what device/dtype the IAE expects
        params = next(iae_model.parameters())
        self.iae_device = params.device
        self.iae_dtype = params.dtype
        
        # 2. Setup Vector (Match IAE)
        self.steering_vec = steering_vec.to(self.iae_device, dtype=self.iae_dtype).view(1, -1)
        self.strength = strength

        # 3. PyVene Flags
        self.is_source_constant = True 
        self.keep_last_dim = True 

    # FIX: Add *args and **kwargs to catch extra inputs
    def forward(self, base, source=None, *args, **kwargs):
        """
        base: The original Gemma activations [Batch, Seq, Hidden_Dim]
        """
        # --- PRE-PROCESSING ---
        # Capture original state to restore later
        orig_device = base.device
        orig_dtype = base.dtype
        orig_shape = base.shape
        
        # 1. Cast and Move to IAE's Domain
        # If Gemma is float16 and IAE is float32, this prevents a crash.
        x_flat = base.reshape(-1, orig_shape[-1]).to(self.iae_device, dtype=self.iae_dtype)

        # --- AUTOENCODER LOGIC ---
        with torch.no_grad():
            # 2. Encode
            z, _ = self.iae.inn(x_flat)
            
            # 3. Split
            bottleneck_dim = self.iae.bottleneck_dim
            y = z[:, :bottleneck_dim]
            z_res = z[:, bottleneck_dim:]
            
            # 4. Steer (Vector is already on iae_device)
            y_steered = y + (self.steering_vec * self.strength)
            
            # 5. Decode
            z_steered = torch.cat([y_steered, z_res], dim=1)
            x_recon, _ = self.iae.inn(z_steered, rev=True)
            
        # --- POST-PROCESSING ---
        # 6. Restore original Device and Dtype
        # If we return float32 to a float16 model, it might crash or slow down.
        return x_recon.to(orig_device, dtype=orig_dtype).view(orig_shape)

In [ ]:
# Load Model
llm = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float32, device_map="cuda")
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [7]:
input_dim = 2304
bottleneck_dim = 128
iae = InvertibleUncertaintyAE(input_dim, bottleneck_dim).cuda()
iae.load_state_dict(model_state_dict)

config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/24.2k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/241M [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

<All keys matched successfully>

In [23]:
# --- CONFIGURATION ---
STEERING_STRENGTH = 10.0

# Instantiate the custom intervention
# We pass the loaded IAE and your latent vector
latent_intervention = LatentSpaceIntervention(
    iae_model=iae,  # Your loaded Autoencoder
    steering_vec=steer_vec, # The 128-dim vector
    strength=STEERING_STRENGTH
)

# PyVene Config
pv_config = pv.IntervenableConfig(
    model_type=type(llm),
    representations=[{
        "layer": layer_idx, # The layer you trained on
        "component": "block_output",
        "unit": "pos",
        "intervention": latent_intervention, 
    }]
)

pv_llm = pv.IntervenableModel(pv_config, llm)

def test_latent_steering(prompt):
    # Prepare Prompt
    if "it" in model_id:
        tuned_prompt = f"<bos><start_of_turn>user\n{prompt}<end_of_turn>\n<start_of_turn>model\n"
    else:
        tuned_prompt = prompt
        
    inputs = tokenizer(tuned_prompt, return_tensors="pt").to("cuda")
    seq_len = inputs["input_ids"].shape[1]

    # 1. Original Generation
    print("Generating Original...")
    orig_output = llm.generate(
        **inputs, 
        max_new_tokens=200,
        do_sample=True,
        temperature=0.7
    )
    orig_text = tokenizer.decode(orig_output[0][seq_len:], skip_special_tokens=True)

    # 2. Steered Generation
    # We target the LAST token position of the prompt to steer the subsequent generation
    print("Generating Steered...")
    _, steered_output = pv_llm.generate(
        inputs,
        # Target the last token (seq_len - 1)
        unit_locations={"base": (None, [[[seq_len - 1]]])}, 
        # Note: We don't need 'source_representations' because the vector 
        # is already baked into our LatentSpaceIntervention class!
        intervene_on_prompt=True,
        max_new_tokens=200,
        do_sample=True,
        temperature=0.7
    )
    steered_text = tokenizer.decode(steered_output[0][seq_len:], skip_special_tokens=True)

    return orig_text, steered_text

In [24]:
# --- RUN ---
test_prompts = [
    "Is 91 a prime number?",
    "How many 's' are there in mississippi?"
]

print(f"Steering Strength: {STEERING_STRENGTH}")
print("-" * 60)

for p in test_prompts:
    orig, steered = test_latent_steering(p)
    print(f"PROMPT: {p}")
    print(f"\n[Original]: {orig}")
    print(f"\n[Steered] : {steered}")
    print("-" * 60)

Steering Strength: 10.0
------------------------------------------------------------
Generating Original...
Generating Steered...
PROMPT: Is 91 a prime number?

[Original]: Yes, 91 is a prime number. Here's why:

* **Definition of a prime number:** A prime number is a whole number greater than 1 that has only two divisors: 1 and itself.

* **Divisibility:**  91 is only divisible by 1 and 91. 
 
Let me know if you have any other numbers you'd like to check! 


[Steered] : Yes, 91 is a prime number. Here's why:

* **Definition of a prime number:** A prime number is a natural number greater than 1 that has only two divisors: 1 and itself.

* **Divisors of 91:** the only divisors of 91 are 1, 7, and 13. 

Since 91 is only divisible by 1 and itself, it meets the criteria for a prime number. 

------------------------------------------------------------
Generating Original...
Generating Steered...
PROMPT: How many 's' are there in mississippi?

[Original]: Let's count them! 

There are **thr